In [0]:
CREATE OR REPLACE TABLE welch.scrap_demo.8d_silver AS
WITH query_results AS (
  SELECT
    path,
    text,
    ai_query(
      'kie-2589d8c7-endpoint',
      text,
      failOnError => false
    ) AS response
  FROM welch.scrap_demo.8d_parsed
)
SELECT
  path,
  parsed_json.*
FROM (
  SELECT
    path,
    from_json(
      CAST(response.result AS STRING),
      'customer_name STRING,
       report_date STRING,
       part_number STRING,
       report_number STRING,
       defect_description STRING,
       quantity_claimed INT,
       root_cause_analysis STRING,
       corrective_actions STRING'
    ) AS parsed_json
  FROM query_results
)

In [0]:
CREATE OR REPLACE TABLE welch.scrap_demo.8d_summarized AS
SELECT
  uuid() as record_id,
  path,
  report_date,
  part_number,
  quantity_claimed,
  ai_summarize(root_cause_analysis) AS rca_summary,
  ai_summarize(corrective_actions) AS ca_summary
FROM welch.scrap_demo.8d_silver;

In [0]:
ALTER TABLE welch.scrap_demo.8d_summarized
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
CREATE OR REPLACE TABLE welch.scrap_demo.scrap_silver AS
SELECT
  uuid() as record_id,
  report_date as scrap_date,
  part_number,
  quantity_claimed

FROM welch.scrap_demo.8d_silver;